# Cutout from a Neuroglancer link

Download a 3D sub-volume from the zarr / precomputed sources behind a Neuroglancer
state link, then view it.


## 1. TLS certificates (internal servers only)

**Skip this cell** if you are using the public example below - it needs no certificate
setup. It is only needed for data served behind an internal certificate authority.

Note that a cutout usually touches two hosts: the state server and the data server. A
bundle holding only your internal CA will break whichever of the two is publicly trusted,
so use one containing both. See the README for the recipe.


In [ ]:
import os

# Path to your internal CA bundle, e.g. "~/certs/rootCA.pem".
# Leave as None if the data host uses a publicly trusted certificate.
CA_BUNDLE = os.environ.get("REQUESTS_CA_BUNDLE")

if CA_BUNDLE:
    CA_BUNDLE = os.path.expanduser(CA_BUNDLE)
    for var in ("CURL_CA_BUNDLE", "SSL_CERT_FILE", "REQUESTS_CA_BUNDLE"):
        os.environ[var] = CA_BUNDLE
    print(f"Using CA bundle: {CA_BUNDLE}")
else:
    print("No CA bundle set - using system defaults")


## 2. Download the cutout

In [ ]:
from ac_data_analysis import cutout_from_NG

# Publicly readable iSPIM dataset - runs as-is, no credentials needed.
# Replace with your own link from the Share button in Neuroglancer.
NG_LINK = (
    "https://neuroglancer-demo.appspot.com/#!"
    "https://apex-connects.s3.us-east-2.amazonaws.com/axonal_connectomics/"
    "contrast_adjusted/H17_PO6_S3_20250422/state2.json"
)

cutout, metadata = cutout_from_NG(
    ng_link=NG_LINK,
    size=(128, 128, 128),   # (z, y, x) in voxels at mip 0
    mip="2",                # note: a string, not an int
    out_path="./cutouts/",
    post_fix="example",
    save_tif=True,
)

print(f"shape {cutout.shape} (x, y, z), dtype {cutout.dtype}")
print(f"coverage {metadata['merge_info']['coverage_ratio']:.1%}")
print(f"sources used: {metadata['merge_info']['sources_used']}")
print(f"saved to {metadata['tif_path']}")


## 3. View it

`microviewer` flips the image to match Neuroglancer's orientation. The cutout is already
transposed to x, y, z, so transpose it back before viewing.


In [ ]:
import microviewer

microviewer.view(cutout.transpose(2, 1, 0), seg=False)


## 4. Midplane

In [ ]:
import matplotlib.pyplot as plt

midplane = cutout[cutout.shape[0] // 2, :, :]
plt.imshow(midplane, cmap="gray")
plt.axis("off")
plt.show()


## 5. Crossing a tile seam

This dataset is tiled into three overlapping sources. A region spanning two of them is
merged according to `merge_strategy`. The default, `first_valid`, picks by source *bounds*
rather than pixel content, so a tile's black edge can win over a neighbour's real data -
leaving a dark band at the seam. Compare the two:


In [ ]:
import numpy as np

# A tall, thin box centred on the src0/src1 seam.
for strategy in ("first_valid", "max"):
    vol, _ = cutout_from_NG(
        ng_link=NG_LINK,
        center=(23304, 12000, 1330),
        size=(128, 8000, 128),
        mip="2",
        merge_strategy=strategy,
        save_tif=False,
    )
    profile = vol.mean(axis=(0, 2))
    print(f"{strategy:12s} all-zero slices at seam: {int((profile == 0).sum())}")


## Optional: contrast enhancement

```python
from skimage import exposure
img_rescale = exposure.equalize_adapthist(cutout, clip_limit=0.03)
```
